# DocuMind — Phase 3B: LayoutLMv3 SROIE Fine-Tuning, Tracking & Deployment Readiness (GPU)

This notebook runs reproducible SROIE preparation, LayoutLMv3 fine-tuning with structured experiment tracking, held-out evaluation, baseline comparison, checkpoint readiness verification, and archive export on Google Colab.

**Runtime:** Select *Runtime -> Change runtime type -> T4 GPU* before running.

In [ ]:
!nvidia-smi
!apt-get update -qq && apt-get install -y -qq tesseract-ocr
!pip install -q torch torchvision transformers datasets seqeval accelerate pymupdf pytesseract opencv-python-headless pillow pydantic pydantic-settings

In [ ]:
import os
if not os.path.exists('DocuMind') and os.path.basename(os.getcwd()) != 'DocuMind':
    !git clone https://github.com/Abhiram-C-Divakaran/DocuMind.git
if os.path.basename(os.getcwd()) != 'DocuMind':
    %cd DocuMind
!git pull --ff-only
!git status -sb

## 1. Prepare Real SROIE Dataset Splits

Downloads `arvindrajan92/sroie_document_understanding` from Hugging Face Hub, persists real RGB receipt images, checks SHA-256 image hashes across splits to prevent leakage, aligns BIO entities (`COMPANY`, `DATE`, `ADDRESS`, `TOTAL`), and writes `data/sroie_processed/`.

In [ ]:
!python -m ml.data.prepare_dataset \
  --hf-dataset arvindrajan92/sroie_document_understanding \
  --output-dir data/sroie_processed \
  --seed 42

with open('data/sroie_processed/validation_report.md') as f:
    print(f.read())

## 2. Fine-Tune `microsoft/layoutlmv3-base`

Defaults to 3 epochs (configurable via `EPOCHS`) with AdamW (`lr=2e-5`), linear warmup (`0.1`), gradient accumulation, CUDA AMP (`fp16`), early stopping, and structured experiment tracking (`ml/experiments/latest_experiment.json`).

In [ ]:
EPOCHS = 3  # Configurable: increase to 6-8 for higher convergence on T4/A100

!python -m ml.training.train_layoutlmv3 \
  --data-dir data/sroie_processed \
  --output-dir checkpoints/layoutlmv3-sroie \
  --best-model-dir checkpoints/layoutlmv3-sroie-best \
  --model-name microsoft/layoutlmv3-base \
  --epochs {EPOCHS} \
  --batch-size 4 \
  --grad-accum 2 \
  --lr 2e-5 \
  --patience 3 \
  --experiment-dir ml/experiments \
  --seed 42

import json
with open('ml/experiments/latest_experiment.json') as f:
    exp = json.load(f)
print('Experiment ID:', exp['experiment_id'])
print('Duration (s):', exp['duration_seconds'])
print('Best checkpoint:', exp['best_checkpoint'])

## 3. Held-Out Test Evaluation, Baseline Benchmark & Serving Verification

In [ ]:
!python -m ml.evaluation.evaluate \
  --checkpoint-dir checkpoints/layoutlmv3-sroie-best \
  --data-dir data/sroie_processed \
  --split test \
  --output-dir ml/evaluation/results

!python -m ml.evaluation.compare_models \
  --checkpoint-dir checkpoints/layoutlmv3-sroie-best \
  --data-dir data/sroie_processed \
  --split test \
  --output-dir ml/evaluation/results \
  --run-full-ocr

from ml.extractors.layoutlmv3_extractor import validate_checkpoint_directory
ok, reason = validate_checkpoint_directory('checkpoints/layoutlmv3-sroie-best')
assert ok, f'Checkpoint validation failed: {reason}'
print('Checkpoint validation passed:', reason)

with open('ml/evaluation/results/evaluation_report.md') as f:
    print(f.read())

with open('ml/evaluation/results/benchmark_comparison.md') as f:
    print(f.read())

## 4. Export Best Checkpoint, Experiment Log & Reports

Archives `checkpoints/layoutlmv3-sroie-best/`, `ml/experiments/latest_experiment.json`, and `ml/evaluation/results/` for local FastAPI serving (`LAYOUTLMV3_CHECKPOINT_DIR=checkpoints/layoutlmv3-sroie-best`).

In [ ]:
!zip -r -q documind_layoutlmv3_best.zip checkpoints/layoutlmv3-sroie-best ml/experiments/latest_experiment.json ml/evaluation/results/evaluation_report.* ml/evaluation/results/benchmark_comparison.*
try:
    from google.colab import files
    files.download('documind_layoutlmv3_best.zip')
except ImportError:
    print('Saved archive at documind_layoutlmv3_best.zip')